# Предварительно обученные модели для векторизации текстовых данных

Здесь не будет библиотеки Spacy, но будут ее составляющие!

In [ ]:
! pip install pymorphy3

## Загрузка данных

Будем использовать [коллекцию постов в Telegram-канале ПМиФИ (до 26.09.2024) ](https://drive.google.com/file/d/1yTXTIghJVvsaJ1foC2xW5W_ruMCOv2Jq/view?usp=sharing).

In [ ]:
import json

In [ ]:
with open('pmifi-tg-26.09.2024.json', 'r', encoding='utf-8') as f:
  raw_json = json.load(f)

In [ ]:
raw_json['messages'][:3]

[{'id': 1,
  'type': 'service',
  'date': '2022-03-10T11:33:46',
  'date_unixtime': '1646890426',
  'actor': 'ПМиФИ ОмГТУ',
  'actor_id': 'channel1687099166',
  'action': 'create_channel',
  'title': 'ПМиФИ',
  'text': '',
  'text_entities': []},
 {'id': 2,
  'type': 'message',
  'date': '2022-03-10T11:44:36',
  'date_unixtime': '1646891076',
  'edited': '2022-03-10T12:03:15',
  'edited_unixtime': '1646892195',
  'from': 'ПМиФИ ОмГТУ',
  'from_id': 'channel1687099166',
  'text': 'Всем добрый день! Будем заселять новые земли!',
  'text_entities': [{'type': 'plain',
    'text': 'Всем добрый день! Будем заселять новые земли!'}]},
 {'id': 3,
  'type': 'service',
  'date': '2022-03-10T11:47:06',
  'date_unixtime': '1646891226',
  'actor': 'ПМиФИ ОмГТУ',
  'actor_id': 'channel1687099166',
  'action': 'edit_group_title',
  'title': 'ПМиФИ ОмГТУ',
  'text': '',
  'text_entities': []}]

In [ ]:
raw_messages = [' '.join(map(lambda x: x['text'], item['text_entities'])) for item in raw_json['messages'] if item['type'] == 'message']
raw_messages = list(filter(lambda x: len(x) > 0, raw_messages))

In [ ]:
raw_messages[:3]

['Всем добрый день! Будем заселять новые земли!',
 'Привет!\n\nВ этом чате мы, кафедра ПМиФИ, делимся полезной информацией, отвечаем на вопросы и приглашаем на наши мероприятия!\n\nВ чате нельзя оскорблять других участников и использовать ненормативную лексику.\n\nПо вопросам поступления обращаться к Виктории Мунько, почта  vkreydunova@mail.ru , тел.  8-923-694-72-55 .',
 'Теперь вопросы и комментарии можно писать в обсуждениях к сообщению.\n\nА уже совсем скоро мы расскажем про различие направлений МО и ФИТ 🔥']

## Предварительная обработка данных

Удалим эмоджи, почты и телефоны с помощью регулярных выражений

In [ ]:
import re

In [ ]:
emoj = re.compile('['
        u'\U0001F600-\U0001F64F'
        u'\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F6FF'
        u'\U0001F1E0-\U0001F1FF'
        u'\U00002500-\U00002BEF'
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        u'\U0001f926-\U0001f937'
        u'\U00010000-\U0010ffff'
        u'\u2640-\u2642'
        u'\u2600-\u2B55'
        u'\u200d'
        u'\u23cf'
        u'\u23e9'
        u'\u231a'
        u'\ufe0f'
        u'\u3030'']+||(\S+@\S+\.\S+)||([+]*[(]{0,1}[0-9]{1,4}[)]{0,1}[-\s\./0-9]*)', re.UNICODE)

In [ ]:
messages = [re.sub(emoj, '', msg) for msg in raw_messages]
messages[:3]

['Всем добрый день! Будем заселять новые земли!',
 'Привет!\n\nВ этом чате мы, кафедра ПМиФИ, делимся полезной информацией, отвечаем на вопросы и приглашаем на наши мероприятия!\n\nВ чате нельзя оскорблять других участников и использовать ненормативную лексику.\n\nПо вопросам поступления обращаться к Виктории Мунько, почта   , тел.  ',
 'Теперь вопросы и комментарии можно писать в обсуждениях к сообщению.\n\nА уже совсем скоро мы расскажем про различие направлений МО и ФИТ ']

Продолжаем предварительную обработку

In [ ]:
messages = [re.sub('[^\s^\w]+', '', msg) for msg in messages]
messages[:3]

['Всем добрый день Будем заселять новые земли',
 'Привет\n\nВ этом чате мы кафедра ПМиФИ делимся полезной информацией отвечаем на вопросы и приглашаем на наши мероприятия\n\nВ чате нельзя оскорблять других участников и использовать ненормативную лексику\n\nПо вопросам поступления обращаться к Виктории Мунько почта    тел  ',
 'Теперь вопросы и комментарии можно писать в обсуждениях к сообщению\n\nА уже совсем скоро мы расскажем про различие направлений МО и ФИТ ']

In [ ]:
messages = [re.sub('\s+', ' ', msg).strip() for msg in messages]
messages[:3]

['Всем добрый день Будем заселять новые земли',
 'Привет В этом чате мы кафедра ПМиФИ делимся полезной информацией отвечаем на вопросы и приглашаем на наши мероприятия В чате нельзя оскорблять других участников и использовать ненормативную лексику По вопросам поступления обращаться к Виктории Мунько почта тел',
 'Теперь вопросы и комментарии можно писать в обсуждениях к сообщению А уже совсем скоро мы расскажем про различие направлений МО и ФИТ']

Нормализизуем и удаляем стоп-слова

In [ ]:
from pymorphy3 import MorphAnalyzer
morph = MorphAnalyzer()

In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
from nltk.tokenize import word_tokenize

In [ ]:
messages = [[morph.normal_forms(w)[0] for w in word_tokenize(msg)] for msg in messages]

In [ ]:
messages[0]

['весь', 'добрый', 'день', 'быть', 'заселять', 'новый', 'земля']

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords
stopwords = stopwords.words('russian')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
messages = [[w for w in msg if w not in stopwords] for msg in messages]
messages[0]

['весь', 'добрый', 'день', 'заселять', 'новый', 'земля']

## Обучение моделей Word2Vec с помощью Gensim

[Gensim](https://radimrehurek.com/gensim/) - библиотека, которая изначально разрабатывалась для решения задач тематического моделирования. Однако с помощью нее удобно обучать модели Word2Vec на собственных данных. При обучении используется алгоритм CBoW.

In [ ]:
from gensim.models import Word2Vec

In [ ]:
w2v = Word2Vec(sentences=messages, min_count=1, vector_size=64, epochs=50)

Векторные представления слов в Gensim имеют тип KeyedVectors. Это важно понимать, поскольку в интернете можно найти много векторных представлений в этом формате. Все они могут быть загружены и использованы с помощью Gensim. Далее мы это увидим.

In [ ]:
w2v.wv

In [ ]:
w2v.wv['кафедра'], w2v.wv['кафедра'].shape

(array([ 1.1970135 ,  0.27954677,  1.7931879 ,  1.2001194 , -0.46640986,
        -1.0334656 ,  0.02003188,  0.19264068,  0.06962533,  0.08357165,
         0.9535911 , -0.5011342 ,  1.3447891 , -1.3116274 , -0.49485418,
         0.86032   , -0.74073654, -0.2670048 ,  0.16266194, -0.01187632,
         1.3446332 ,  1.4018337 ,  1.6265377 , -1.042916  , -0.3933845 ,
         0.952019  ,  0.2589134 ,  0.8632862 ,  1.2376739 , -0.283387  ,
        -0.4734974 ,  0.04684284, -0.7372467 , -1.0732448 , -0.79425406,
        -0.47696447,  0.42541203,  0.26106802,  1.5026951 ,  0.5116206 ,
         0.3482097 , -0.6951472 , -0.9384277 , -0.3999327 ,  0.8224524 ,
         0.59423226,  0.7904438 , -1.6063657 , -0.6662436 ,  0.49320483,
         0.48333883,  0.5132476 , -1.2624708 ,  1.1320655 ,  1.7105037 ,
        -0.53479654,  0.64097816, -0.70537925, -0.01439023, -0.5335458 ,
         0.08686632, -0.16586243,  0.37861294,  0.43930382], dtype=float32),
 (64,))

А вот здесь начинается магия!

In [ ]:
w2v.wv.most_similar('кафедра', topn=3)

[('преподаватель', 0.9158676266670227),
 ('реклама', 0.8941570520401001),
 ('приглашать', 0.8858376741409302)]

In [ ]:
w2v.wv.most_similar('конференция', topn=5)

[('научнопрактический', 0.9124317169189453),
 ('жарка', 0.894753098487854),
 ('молодёжный', 0.8837294578552246),
 ('xiii', 0.8697914481163025),
 ('xiv', 0.8635475039482117)]

Можем сохранить нашу модель Word2Vec, чтобы использовать ее в будущем

In [ ]:
w2v.save('myw2v.bin')

In [ ]:
w2v2 = Word2Vec.load('myw2v.bin')
w2v2.wv.most_similar('конференция', topn=5)

[('научнопрактический', 0.9124317169189453),
 ('жарка', 0.894753098487854),
 ('молодёжный', 0.8837294578552246),
 ('xiii', 0.8697914481163025),
 ('xiv', 0.8635475039482117)]

## Предварительно обученные модели RusVectores

Для русского и украинского языка существууют предварительно обученные модели [RusVectores](https://rusvectores.org/ru/models/)

Самую известную модель с векторами размерности 300, обученную на уже известном вам корпусе RuCorpora, можно загрузить используя API Gensim. Другие модели можно скачать с сайта и загрузить с диска в Gensim, как было показано выше.

С помощью API Gensim можно скачать самые разные модели

In [ ]:
from gensim.downloader import load, info

In [ ]:
list(info()['models'].keys())

['fasttext-wiki-news-subwords-300',
 'conceptnet-numberbatch-17-06-300',
 'word2vec-ruscorpora-300',
 'word2vec-google-news-300',
 'glove-wiki-gigaword-50',
 'glove-wiki-gigaword-100',
 'glove-wiki-gigaword-200',
 'glove-wiki-gigaword-300',
 'glove-twitter-25',
 'glove-twitter-50',
 'glove-twitter-100',
 'glove-twitter-200',
 '__testing_word2vec-matrix-synopsis']

In [ ]:
rv = load('word2vec-ruscorpora-300')

[==================================================] 100.0% 198.8/198.8MB downloaded


In [ ]:
rv

Обратите внимание, что тип данных - KeyedVectors. Ранее мы оборачивали этот тип в объект Word2Vec.

In [ ]:
rv.most_similar('кафедра_NOUN')

[('доцент_NOUN', 0.6663437485694885),
 ('декан_NOUN', 0.6478123664855957),
 ('профессор_NOUN', 0.6361485123634338),
 ('факультет_NOUN', 0.6278356909751892),
 ('университет_NOUN', 0.6136446595191956),
 ('преподаватель_NOUN', 0.6078613996505737),
 ('ректор_NOUN', 0.6058201789855957),
 ('мирэа_NOUN', 0.5775336027145386),
 ('тихоновский::гуманитарный_ADJ', 0.5705974102020264),
 ('адъюнкт-профессор_NOUN', 0.5699172616004944)]

In [ ]:
rv['кафедра_NOUN'].shape

(300,)

In [ ]:
import numpy as np
rv.similar_by_vector(np.zeros(300))

[('пахтать_VERB', 0.0),
 ('свидетель::платайс_NOUN', 0.0),
 ('амфиктион_NOUN', 0.0),
 ('эйс_NOUN', 0.0),
 ('германос_NOUN', 0.0),
 ('етоты_NOUN', 0.0),
 ('козырева_NOUN', 0.0),
 ('микроген_NOUN', 0.0),
 ('натюрель_NOUN', 0.0),
 ('ахмедова::щель_NOUN', 0.0)]

## Предварительно обученные модели FastText

FastText - это способ организации векторного пространства для текстовых данных, при котором мы сопоставляем вектора не словам, а n-граммам символов, из которых они состоят. Векторное представление слова строится как среднее значение векторов входящих в него n-грамм.

Для обучения и использования моделей с использованием данного подхода существует библиотека [fasttext](https://fasttext.cc/)

In [ ]:
! pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp310-cp310-linux_x86_64.whl size=4296188 sha256=e8be2436dfd1392ece723ec65edd9344a59f2724551926840a94afd3df9feabd
  Stored in directory: /root/.cache/pip/wheels/0d/a2/00/81db54d3e6a8199b829d58e02cec2ddb20ce3e59fad8d3c92a
Successfully built fasttext


Как и Gensim библиотека fasttext позволяет обучать собственные модели Word2Vec, однако в данном случае мы посмотрим на предварительно обученную модель для русского языка

In [ ]:
from fasttext.util import download_model
import fasttext

Модель для русского языка весит 4 ГБ.

In [ ]:
#ft_pre = fasttext.load_model(download_model('ru'))

In [ ]:
a = ' '.join([' '.join(msg) for msg in messages])
with open('all_messages_text.txt', 'w') as f:
  f.write(a)

In [ ]:
ft = fasttext.train_unsupervised('all_messages_text.txt', model='skipgram', dim=64)

In [ ]:
ft.get_nearest_neighbors('математика', k=4)

[(0.9999967217445374, 'математик'),
 (0.9999929070472717, 'математический'),
 (0.9999911785125732, 'физикоматематический'),
 (0.9999886155128479, 'информатика')]

In [ ]:
ft.get_word_vector('кафедра').shape

(64,)

In [ ]:
ft.save_model('ft.bin')

## Предварительно обученная модель Navec и проект Natasha

В 2020 году [Лаборатория анализа данных Александра Кукушкина](https://lab.alexkuk.ru/) представила проект с открытым исходным кодом [Natasha](https://github.com/natasha/natasha).

Проект объединил в себе современные инструменты для обработки русского языка (токенизатор [Razdel](https://github.com/natasha/razdel), векторизатор [Navec](https://github.com/natasha/navec), размеченный для решения морфологических задач набор данных [Nerus](https://github.com/natasha/nerus) и др.)

**Все эти инструменты используются внутри моделей для русского языка фреймворка Spacy!**

Поэтому их можно использовать отдельно друг от друга. Razdel работает для русского языка лучше, чем стандартный токенизатор из NLTK. Рекомендуется использовать его

In [ ]:
! pip install razdel

In [ ]:
from razdel import tokenize

In [ ]:
tokens = list(tokenize('Где-то на высоте 5 м кубических, площади 3м 2'))
tokens

[Substring(0, 6, 'Где-то'),
 Substring(7, 9, 'на'),
 Substring(10, 16, 'высоте'),
 Substring(17, 18, '5'),
 Substring(19, 20, 'м'),
 Substring(21, 31, 'кубических'),
 Substring(31, 32, ','),
 Substring(33, 40, 'площади'),
 Substring(41, 42, '3'),
 Substring(42, 43, 'м'),
 Substring(44, 45, '2')]

In [ ]:
tokens[0]

Substring(0, 6, 'Где-то')

In [ ]:
tokens[0].text

'Где-то'

Займемся векторизацией

In [ ]:
! pip install navec

In [ ]:
! wget https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar

--2024-09-26 05:08:46--  https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53012480 (51M) [application/x-tar]
Saving to: ‘navec_hudlit_v1_12B_500K_300d_100q.tar’

navec_hudlit_v1_12B 100%[===================>]  50.56M  16.5MB/s    in 3.5s    

2024-09-26 05:08:51 (14.5 MB/s) - ‘navec_hudlit_v1_12B_500K_300d_100q.tar’ saved [53012480/53012480]



In [ ]:
from navec import Navec

In [ ]:
navec = Navec.load('navec_hudlit_v1_12B_500K_300d_100q.tar')

In [ ]:
navec['кафедра'].shape

(300,)

In [ ]:
'мы' in navec, 'пмифи' in navec

(True, False)

Предусмотрен вектор для неизвестных слов

In [ ]:
navec['<unk>'].shape

(300,)

Также предусмотрен вектор из нулей для использования при заполнении последовательностей (про это мы еще поговорим позже)

In [ ]:
navec['<pad>'].shape, sum(navec['<pad>'])

((300,), 0.0)

Определяем семантическую близость слов (косинус между векторами)

In [ ]:
navec.sim('профессор', 'ректор')

0.6199871

Теперь вы можете использовать при решении своих задач (в том числе с применением рекуррентных нейронных сетей) предварительно обученные векторные представления.

Далее в курсе NLP мы еще вернемся к векторизации текстов, будем говорить уже про векторизацию предложений. Но сначала нужно будет познакомиться с такой архитектурой нейронных сетей как трансформер.